## HyDE(Hypothetical Document Embeddings)를 사용한 RAG

HyDE는 사용자 질문을 그대로 임베딩하지 않고, **LLM이 질문에 대한 가상(hypothetical) 답변 문서를 먼저 생성**한 뒤 그 문서를 임베딩하여 검색하는 기법이다.  
질문과 문서의 문체·표현 차이를 줄여, 짧은/모호한 질의에서도 관련 문서를 더 잘 찾을 수 있다.

**논문:** [Precise Zero-Shot Dense Retrieval without Relevance Labels (Gao et al., 2022)](https://arxiv.org/abs/2212.10496)

### 흐름
1. 사용자 질문 입력
2. LLM이 "이 질문에 답하는 가상의 문서" 생성
3. 가상 문서를 임베딩
4. 벡터 검색으로 실제 문서 검색
5. 검색된 실제 문서로 최종 답변 생성

본 노트북에서는 **일반 코드 구현**과 **LangChain `HypotheticalDocumentEmbedder`** 구현을 모두 다룬다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### [0] 공통 준비: LLM, 임베딩, 샘플 문서, 벡터스토어

In [2]:
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# HyDE 효과를 보기 좋은 샘플 코퍼스
# (질문 표현과 문서 표현이 다를 수 있도록 서술형으로 작성)
docs = [
    Document(
        page_content=(
            "하이브리드 검색은 키워드 기반의 희소 검색(BM25)과 의미 기반의 밀집 검색(벡터 검색)을 "
            "결합하여 관련 문서를 찾는 방식이다. BM25는 정확한 용어 매칭에 강하고, 벡터 검색은 "
            "동의어·문맥적 유사성에 강하다. 두 결과를 점수 정규화 후 가중 합산하거나 RRF로 융합한다."
        ),
        metadata={"topic": "hybrid_search"},
    ),
    Document(
        page_content=(
            "리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 "
            "상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. "
            "Cohere Rerank, bge-reranker 등이 대표적이다."
        ),
        metadata={"topic": "reranker"},
    ),
    Document(
        page_content=(
            "청킹(Chunking)은 긴 문서를 검색·임베딩에 적합한 크기로 나누는 과정이다. "
            "청크가 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 검색 품질이 떨어진다. "
            "RecursiveCharacterTextSplitter처럼 구분자 우선순위를 두는 방식이 널리 쓰인다."
        ),
        metadata={"topic": "chunking"},
    ),
    Document(
        page_content=(
            "쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 기법이다. "
            "Multi-Query는 질문을 다양한 관점으로 재작성하고, HyDE는 가상의 답변 문서를 생성해 "
            "그 임베딩으로 검색한다. 짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 된다."
        ),
        metadata={"topic": "query_expansion"},
    ),
    Document(
        page_content=(
            "임베딩 모델은 텍스트를 고차원 벡터로 변환한다. text-embedding-3-small은 비용 대비 "
            "성능이 좋은 OpenAI 임베딩이다. 도메인 특화 문서에서는 파인튜닝 또는 도메인 적응형 "
            "임베딩이 검색 품질을 개선할 수 있다."
        ),
        metadata={"topic": "embedding"},
    ),
    Document(
        page_content=(
            "RAGAS는 RAG 파이프라인의 응답 품질을 평가하는 프레임워크이다. faithfulness, "
            "answer relevancy, context precision, context recall 등의 지표로 검색·생성 품질을 "
            "정량화한다."
        ),
        metadata={"topic": "evaluation"},
    ),
]

vectorstore = FAISS.from_documents(docs, embeddings)
print(f"벡터스토어 문서 수: {len(docs)}")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_25224\231984244.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


벡터스토어 문서 수: 6


### [1] 베이스라인: Naive RAG (질문 직접 임베딩)

질문을 그대로 임베딩하여 유사 문서를 검색한 뒤 답변을 생성한다.

In [3]:
def format_docs(documents: List[Document]) -> str:
    return "\n\n".join(
        f"[{i+1}] ({d.metadata.get('topic', '-')}) {d.page_content}"
        for i, d in enumerate(documents)
    )


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 문맥만 근거로 질문에 답하라. 문맥에 없으면 모른다고 말하라.\n\n[문맥]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
answer_chain = answer_prompt | llm | StrOutputParser()


def naive_rag(question: str, k: int = 3) -> dict:
    retrieved = vectorstore.similarity_search(question, k=k)
    answer = answer_chain.invoke(
        {"context": format_docs(retrieved), "question": question}
    )
    return {"retrieved": retrieved, "answer": answer}


query = "짧은 질문이랑 문서 표현이 다를 때 검색을 어떻게 개선해?"

naive_result = naive_rag(query)
print("=== Naive RAG 검색 결과 ===")
print(format_docs(naive_result["retrieved"]))
print("\n=== Naive RAG 답변 ===")
print(naive_result["answer"])

=== Naive RAG 검색 결과 ===
[1] (query_expansion) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 기법이다. Multi-Query는 질문을 다양한 관점으로 재작성하고, HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색한다. 짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 된다.

[2] (chunking) 청킹(Chunking)은 긴 문서를 검색·임베딩에 적합한 크기로 나누는 과정이다. 청크가 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 검색 품질이 떨어진다. RecursiveCharacterTextSplitter처럼 구분자 우선순위를 두는 방식이 널리 쓰인다.

[3] (reranker) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

=== Naive RAG 답변 ===
짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 되는 방법으로는 HyDE가 있습니다. HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색하는 기법입니다. 이를 통해 검색 품질을 개선할 수 있습니다.


### [2] 일반 코드로 HyDE RAG 구현

핵심 아이디어:
1. LLM에게 "질문에 답하는 가상의 문서"를 생성하게 한다
2. **가상 문서**를 임베딩하여 벡터 검색
3. 검색된 **실제 문서**로 최종 답변 생성

> 주의: 가상 문서는 검색용일 뿐, 최종 답변의 근거로 사용하지 않는다.

In [4]:
# 가상 문서 생성 프롬프트
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "너는 검색 품질을 높이기 위한 가상 문서를 작성하는 도우미다.\n"
                "주어진 질문에 대해, 실제 지식 문서처럼 보이는 짧은 단락(2~4문장)을 작성하라.\n"
                "사실 여부를 과도하게 검증하지 말고, 질문에 대한 답변 형태의 서술문을 생성하라.\n"
                "제목이나 불릿 없이 본문만 출력하라."
            ),
        ),
        ("human", "질문: {question}"),
    ]
)

hyde_generator = hyde_prompt | llm | StrOutputParser()


def generate_hypothetical_document(question: str) -> str:
    """질문에 대한 가상(hypothetical) 문서를 생성한다."""
    return hyde_generator.invoke({"question": question})


def hyde_retrieve_manual(question: str, k: int = 3) -> dict:
    """일반 코드로 구현한 HyDE 검색."""
    # 1) 가상 문서 생성
    hypo_doc = generate_hypothetical_document(question)

    # 2) 가상 문서로 유사도 검색 (질문 대신 가상 문서 사용)
    retrieved = vectorstore.similarity_search(hypo_doc, k=k)

    return {"hypothetical_document": hypo_doc, "retrieved": retrieved}


def hyde_rag_manual(question: str, k: int = 3) -> dict:
    """일반 코드로 구현한 HyDE RAG (검색 + 답변 생성)."""
    retrieval = hyde_retrieve_manual(question, k=k)
    answer = answer_chain.invoke(
        {
            "context": format_docs(retrieval["retrieved"]),
            "question": question,
        }
    )
    return {**retrieval, "answer": answer}


manual_result = hyde_rag_manual(query)

print("=== [수동 HyDE] 생성된 가상 문서 ===")
print(manual_result["hypothetical_document"])
print("\n=== [수동 HyDE] 검색 결과 ===")
print(format_docs(manual_result["retrieved"]))
print("\n=== [수동 HyDE] 최종 답변 ===")
print(manual_result["answer"])

=== [수동 HyDE] 생성된 가상 문서 ===
검색 품질을 개선하기 위해서는 사용자의 질문 의도를 이해하고, 이를 기반으로 관련된 문서 표현을 매칭하는 알고리즘을 개발하는 것이 중요하다. 자연어 처리 기술을 활용하여 질문의 의미를 분석하고, 유사한 맥락의 문서들을 추천함으로써 사용자가 원하는 정보를 더 쉽게 찾을 수 있도록 해야 한다. 또한, 사용자 피드백을 반영하여 검색 결과를 지속적으로 개선하는 것도 효과적이다.

=== [수동 HyDE] 검색 결과 ===
[1] (query_expansion) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 기법이다. Multi-Query는 질문을 다양한 관점으로 재작성하고, HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색한다. 짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 된다.

[2] (hybrid_search) 하이브리드 검색은 키워드 기반의 희소 검색(BM25)과 의미 기반의 밀집 검색(벡터 검색)을 결합하여 관련 문서를 찾는 방식이다. BM25는 정확한 용어 매칭에 강하고, 벡터 검색은 동의어·문맥적 유사성에 강하다. 두 결과를 점수 정규화 후 가중 합산하거나 RRF로 융합한다.

[3] (evaluation) RAGAS는 RAG 파이프라인의 응답 품질을 평가하는 프레임워크이다. faithfulness, answer relevancy, context precision, context recall 등의 지표로 검색·생성 품질을 정량화한다.

=== [수동 HyDE] 최종 답변 ===
짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 되는 방법으로는 쿼리 확장(query expansion) 기법을 사용할 수 있습니다. 이 기법은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 데 도움을 줍니다.


### [3] LangChain `HypotheticalDocumentEmbedder`로 구현

LangChain v1 이후 HyDE는 `langchain-classic` 패키지에 있다.

```text
pip install langchain-classic
```

`HypotheticalDocumentEmbedder`는 임베딩 인터페이스를 구현한다.
- `embed_query()`: 질문 → 가상 문서 생성 → 임베딩 반환
- 문서 인덱싱(`embed_documents`)은 기존 `base_embeddings`를 그대로 사용

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import HypotheticalDocumentEmbedder

# 커스텀 프롬프트 (입력 변수명은 question)
custom_hyde_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "다음 질문에 답하는 것처럼 보이는 짧은 지식 문서 단락을 작성하라.\n"
        "2~4문장, 서술형, 제목/불릿 없이 본문만 출력하라.\n\n"
        "질문: {question}\n\n"
        "가상 문서:"
    ),
)

# HyDE 임베더 생성
# - prompt_key 대신 custom_prompt 사용
# - 기본 prompt_key 예: "web_search", "sci_fact", "fiqa", "trec_news" 등
hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    custom_prompt=custom_hyde_prompt,
)

print("HypotheticalDocumentEmbedder 생성 완료")
print(f"타입: {type(hyde_embeddings)}")

HypotheticalDocumentEmbedder 생성 완료
타입: <class 'langchain_classic.chains.hyde.base.HypotheticalDocumentEmbedder'>


In [7]:
import numpy as np


def peek_hypothetical_document(hyde_embedder, question: str) -> str:
    """HyDE 임베더 내부의 가상 문서 텍스트를 확인한다.

    프롬프트마다 입력 변수명이 다르므로(custom_prompt는 question,
    내장 prompt_key는 QUESTION) input_keys에서 변수명을 가져온다.
    """
    var_name = hyde_embedder.input_keys[0]
    return hyde_embedder.llm_chain.invoke({var_name: question})


def hyde_retrieve_langchain(question: str, k: int = 3) -> dict:
    """LangChain HypotheticalDocumentEmbedder로 HyDE 검색."""
    # 1) 가상 문서 텍스트도 함께 확인 (디버깅/학습용)
    hypo_doc = peek_hypothetical_document(hyde_embeddings, question)

    # 2) HyDE 임베딩으로 질의 벡터 생성
    query_vector = hyde_embeddings.embed_query(question)

    # 3) 기존 벡터스토어에 벡터로 유사도 검색
    #    (인덱스는 일반 임베딩으로 구축되어 있음)
    retrieved = vectorstore.similarity_search_by_vector(query_vector, k=k)

    return {
        "hypothetical_document": hypo_doc,
        "query_vector_dim": len(query_vector),
        "query_vector_norm": float(np.linalg.norm(query_vector)),
        "retrieved": retrieved,
    }


def hyde_rag_langchain(question: str, k: int = 3) -> dict:
    """LangChain HypotheticalDocumentEmbedder 기반 HyDE RAG."""
    retrieval = hyde_retrieve_langchain(question, k=k)
    answer = answer_chain.invoke(
        {
            "context": format_docs(retrieval["retrieved"]),
            "question": question,
        }
    )
    return {**retrieval, "answer": answer}


lc_result = hyde_rag_langchain(query)

print("=== [LangChain HyDE] 생성된 가상 문서 ===")
print(lc_result["hypothetical_document"])
print(f"\n쿼리 벡터 차원: {lc_result['query_vector_dim']}")
print(f"쿼리 벡터 L2 norm: {lc_result['query_vector_norm']:.4f}")
print("\n=== [LangChain HyDE] 검색 결과 ===")
print(format_docs(lc_result["retrieved"]))
print("\n=== [LangChain HyDE] 최종 답변 ===")
print(lc_result["answer"])

=== [LangChain HyDE] 생성된 가상 문서 ===
검색 개선을 위해서는 사용자의 질문 의도를 파악하고, 다양한 표현을 인식할 수 있는 알고리즘을 개발하는 것이 중요하다. 이를 위해 자연어 처리 기술을 활용하여 질문의 의미를 분석하고, 유사한 문장 구조나 동의어를 포함한 검색 결과를 제공해야 한다. 또한, 사용자 피드백을 반영하여 검색 결과의 품질을 지속적으로 향상시키는 것도 필요하다.

쿼리 벡터 차원: 1536
쿼리 벡터 L2 norm: 1.0000

=== [LangChain HyDE] 검색 결과 ===
[1] (query_expansion) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 기법이다. Multi-Query는 질문을 다양한 관점으로 재작성하고, HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색한다. 짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 된다.

[2] (hybrid_search) 하이브리드 검색은 키워드 기반의 희소 검색(BM25)과 의미 기반의 밀집 검색(벡터 검색)을 결합하여 관련 문서를 찾는 방식이다. BM25는 정확한 용어 매칭에 강하고, 벡터 검색은 동의어·문맥적 유사성에 강하다. 두 결과를 점수 정규화 후 가중 합산하거나 RRF로 융합한다.

[3] (reranker) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

=== [LangChain HyDE] 최종 답변 ===
짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 되는 기법으로는 HyDE가 있습니다. HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색하는 방법을 사용합니다.


In [8]:
lc_result

{'hypothetical_document': '검색 개선을 위해서는 사용자의 질문 의도를 파악하고, 다양한 표현을 인식할 수 있는 알고리즘을 개발하는 것이 중요하다. 이를 위해 자연어 처리 기술을 활용하여 질문의 의미를 분석하고, 유사한 문장 구조나 동의어를 포함한 검색 결과를 제공해야 한다. 또한, 사용자 피드백을 반영하여 검색 결과의 품질을 지속적으로 향상시키는 것도 필요하다.',
 'query_vector_dim': 1536,
 'query_vector_norm': 0.9999643029677359,
 'retrieved': [Document(id='703c81fd-0a2e-4ebc-8749-58bb50107249', metadata={'topic': 'query_expansion'}, page_content='쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓히는 기법이다. Multi-Query는 질문을 다양한 관점으로 재작성하고, HyDE는 가상의 답변 문서를 생성해 그 임베딩으로 검색한다. 짧은 질문과 문서 표현 사이의 간극을 줄이는 데 도움이 된다.'),
  Document(id='40cff9f9-6672-43bc-ab46-e33b18c03cae', metadata={'topic': 'hybrid_search'}, page_content='하이브리드 검색은 키워드 기반의 희소 검색(BM25)과 의미 기반의 밀집 검색(벡터 검색)을 결합하여 관련 문서를 찾는 방식이다. BM25는 정확한 용어 매칭에 강하고, 벡터 검색은 동의어·문맥적 유사성에 강하다. 두 결과를 점수 정규화 후 가중 합산하거나 RRF로 융합한다.'),
  Document(id='5aba728f-c602-4aee-90cf-1f3bbcdcd493', metadata={'topic': 'reranker'}, page_content='리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면

In [9]:
query

'짧은 질문이랑 문서 표현이 다를 때 검색을 어떻게 개선해?'

### [4] 내장 prompt_key 사용 예제

`custom_prompt` 대신 논문/벤치마크용 기본 프롬프트 키를 쓸 수도 있다.

> **주의:** 내장 프롬프트의 입력 변수명은 소문자 `question`이 아니라 대문자 **`QUESTION`** 이다.  
> `llm_chain`을 직접 호출할 때 `{"question": ...}`을 넘기면 `KeyError`가 발생하므로,
> `input_keys[0]`로 변수명을 가져와 사용한다.  
> (`embed_query()`는 내부에서 알아서 처리하므로 영향이 없다.)

In [9]:
# 기본 프롬프트 키로 HyDE 임베더 생성
hyde_web = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key="web_search",  # 또는 sci_fact, fiqa, trec_news 등
)

# 내장 프롬프트는 입력 변수명이 대문자 QUESTION이다.
print(f"내장 프롬프트 입력 변수: {hyde_web.input_keys}")

web_query = "How can short user queries retrieve better matching documents?"
hypo_en = peek_hypothetical_document(hyde_web, web_query)
web_vector = hyde_web.embed_query(web_query)
web_docs = vectorstore.similarity_search_by_vector(web_vector, k=3)

print("\n=== prompt_key='web_search' 가상 문서 ===")
print(hypo_en)
print("\n=== 검색 결과 ===")
print(format_docs(web_docs))

내장 프롬프트 입력 변수: ['QUESTION']

=== prompt_key='web_search' 가상 문서 ===
Short user queries can retrieve better matching documents by leveraging advanced search algorithms and natural language processing techniques that focus on the core intent of the query. When users input concise phrases or keywords, search engines can quickly analyze and match these terms against indexed content, prioritizing relevance and context. 

Additionally, short queries often eliminate unnecessary words, allowing the search engine to hone in on the most critical elements of the user's request. This streamlined approach can enhance the precision of search results, as algorithms can more effectively identify documents that contain the key terms or concepts. 

Moreover, many search engines utilize machine learning to understand user behavior and preferences, which helps refine results based on past interactions. By analyzing patterns in short queries, these systems can improve their ability to predict what users are

### [5] Naive vs HyDE 비교

여러 질문에 대해 Naive RAG와 HyDE RAG의 검색 topic을 나란히 비교한다.

In [10]:
test_queries = [
    "짧은 질문이랑 문서 표현이 다를 때 검색을 어떻게 개선해?",
    "키워드 검색이랑 의미 검색을 같이 쓰는 방법이 뭐야?",
    "1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?",
]


def topics(documents: List[Document]) -> List[str]:
    return [d.metadata.get("topic", "-") for d in documents]


print(f"{'질문':<40} | {'Naive':<28} | {'HyDE(수동)'}")
print("-" * 110)

for q in test_queries:
    naive_docs = vectorstore.similarity_search(q, k=2)
    hyde_docs = hyde_retrieve_manual(q, k=2)["retrieved"]

    print(
        f"{q[:38]:<40} | {str(topics(naive_docs)):<28} | {topics(hyde_docs)}"
    )

질문                                       | Naive                        | HyDE(수동)
--------------------------------------------------------------------------------------------------------------
짧은 질문이랑 문서 표현이 다를 때 검색을 어떻게 개선해?         | ['query_expansion', 'chunking'] | ['query_expansion', 'reranker']
키워드 검색이랑 의미 검색을 같이 쓰는 방법이 뭐야?            | ['hybrid_search', 'reranker'] | ['hybrid_search', 'reranker']
1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?        | ['reranker', 'chunking']     | ['reranker', 'hybrid_search']


### [6] 정리

| 구분 | Naive RAG | HyDE RAG |
|---|---|---|
| 검색 벡터 | 질문 임베딩 | 가상 문서 임베딩 |
| 장점 | 단순·빠름 | 질문-문서 표현 간극 완화 |
| 단점 | 짧은/모호한 질의에 취약 | LLM 호출 비용·지연 증가 |
| 주의 | - | 가상 문서는 검색용, 최종 근거로 쓰지 않음 |

**구현 포인트**
- 일반 구현: `가상문서 생성 → similarity_search(가상문서) → 실제 문서로 답변`
- LangChain: `HypotheticalDocumentEmbedder.embed_query()` → `similarity_search_by_vector()`
- LangChain v1에서는 `from langchain_classic.chains import HypotheticalDocumentEmbedder`

**언제 쓰면 좋은가**
- 질문이 짧거나 키워드가 부족한 경우
- 사용자 표현과 문서 문체가 크게 다른 경우
- Multi-Query와 함께 쿼리 변환 전략의 하나로 라우팅할 때